# Notebook 1: wound (hole) tracking

Finds a laser wound (a dark hole in the membrane channel) and tracks its centre and radius in every frame as it
closes. Notebook 2 uses this tracking to measure the Ca²⁺ wave at fixed distances from the moving wound edge.

**Quick start (example data).** Run all cells (in Jupyter: *Run → Run All Cells*). The notebook is set up for the
example dataset ctrl_0820 with the settings used in the paper. Expected result: the wound radius goes from 40 px at
frame 21 to 10 px at frame 64, saved in `wound_tracking/ctrl_0820_membrane/hole_tracking_results_new.csv`. Then
run notebook 2.

**Your own data.** Follow the checklist in section 1.

**Contents**
1. Settings to edit (with the checklist for your own data)
2. Method settings (the values used in the paper)
3. Load the movie and check the seed
4. The two tracking methods
5. Run the tracking
6. Check the result
7. Save
8. Troubleshooting

Appendix: settings used for the paper's datasets.

**Output.** `wound_tracking/<membrane folder>/hole_tracking_results_new.csv`, one row per frame:
`frame, cy, cx, radius` (pixels; `cy` = row, `cx` = column; NaN before `START_FRAME`). The supplied tracking of
each dataset (`hole_tracking_results.csv`) is never overwritten.

Paths: run this notebook from the `notebooks/` folder. Data and results folders default to `data/` and `results/` in the repository and can be changed with the environment variables listed in README.md.

## 1. Settings to edit

The next cell holds everything you normally need to change. It is set up for the example data.

### Using your own data: checklist

1. **Images.** One folder per movie with **one 2-D, single-channel TIFF per frame**, named so that sorting by file
   name gives time order (e.g. `frame0000.tif`, `frame0001.tif`, ...). Use the channel in which the wound is a
   dark hole (here: a membrane marker). Put the folder in `data/wound/` (or point `WOUND_DATA_DIR` to where it
   is) and set `MEMBRANE_FOLDER` to its name. If your files are not `.tif`, set `MEMBRANE_IMAGE_GLOB`
   (e.g. `"*.tiff"`). A multi-frame TIFF stack must first be saved as one file per frame (e.g. in Fiji:
   *File → Save As → Image Sequence*).
2. **`START_FRAME`**: the first frame in which the wound is clearly visible as a dark hole.
3. **`INIT_ROI`**: a box `(y0, y1, x0, x1)` in pixels (rows `y0`-`y1`, columns `x0`-`x1`, measured from the
   top-left corner) that contains the hole on `START_FRAME` and as little other dark area as possible. Section 3
   draws the box on the image: run up to there, adjust, and run again. Keep `SEED = "detect"`.
4. **`METHOD`**: `"halfway"` follows a hole that fills in and fades as it closes; `"otsu"` is the earlier method,
   which works while the hole stays clearly darker than the tissue (section 4). If one fails on your movie, try
   the other.
5. **Run all cells**, then check the figures in section 6: the outline should sit on the hole edge in every frame
   and the radius should shrink smoothly. If not, see section 8.
6. In notebook 2, set `TRACKING_CSV = "hole_tracking_results_new.csv"` (the default) and the dataset's folders.

The tracking works in pixels and frames; the pixel size and frame interval matter only in notebook 2.

In [ ]:
from pathlib import Path
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile

warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# SETTINGS TO EDIT (set up for the example data, ctrl_0820, with the settings used in the paper)
# =============================================================================

# ---- where the data are (see README.md; environment variables override these defaults) ----
REPO_ROOT = Path.cwd().parent   # this notebook runs from <repository>/notebooks
DATA_ROOT = Path(os.environ.get("WOUND_DATA_DIR", REPO_ROOT / "data" / "wound"))
TRACKING_ROOT = Path(os.environ.get("WOUND_TRACKING_DIR", REPO_ROOT / "wound_tracking"))

# ---- the movie ----
MEMBRANE_FOLDER = "ctrl_0820_membrane"   # folder in DATA_ROOT with one TIFF per frame
MEMBRANE_IMAGE_GLOB = "*.tif"            # which files in that folder are the frames (sorted by name = time order)

# ---- tracking method: "halfway" or "otsu" (section 4) ----
METHOD = "halfway"

# ---- seed: the wound circle on START_FRAME, where tracking starts ----
START_FRAME = 21                  # first frame in which the wound is clearly visible as a dark hole
SEED = "detect"                   # "detect": find the hole inside INIT_ROI (use this for new data)
                                  # "saved":  circle of the supplied hole_tracking_results.csv ("halfway" only)
INIT_ROI = (140, 270, 225, 360)   # box around the hole on START_FRAME: (y0, y1, x0, x1) in pixels
                                  # (rows y0-y1, columns x0-x1); section 3 draws it on the image

# ---- output: TRACKING_ROOT/<MEMBRANE_FOLDER>/<HOLE_TRACKING_CSV>, read by notebook 2 ----
HOLE_TRACKING_CSV = "hole_tracking_results_new.csv"


## 2. Method settings (the values used in the paper)

These control how each method finds the hole edge (section 4 explains the methods). The values below were used
for every dataset in the paper; change them only if the tracking fails (section 8 says which to try). The cell
also checks the settings and finds the image files.

In [ ]:
# =============================================================================
# METHOD SETTINGS: the values used in the paper. Change them only if the tracking fails (section 8).
# =============================================================================

# ---- "halfway" method ----
CLIP_PERCENTILE = 95         # every frame is capped at this percentile before tracking, so that a very bright spot
                             # near the wound does not set the seed threshold
TRACK_SIGMA = 3.0            # Gaussian blur (px) before thresholding
TRACK_LEVEL = 0.5            # edge level: floor + TRACK_LEVEL x (tissue - floor)
TRACK_TISSUE_RING = (8, 30)  # tissue level = median in this ring outside the previous edge (px)
TRACK_WINDOW = (0.3, 6)      # search window: previous radius + max(0.3 x radius, 6 px)
TRACK_MAX_JUMP = 8           # largest centre move between frames (px); otherwise the previous circle is held

# ---- "otsu" method ----
SWITCH_FRAME = 41            # from this frame on the search window shrinks with the hole
SEARCH_RADIUS_EARLY = 55     # half-size of the search window before SWITCH_FRAME (px)
LATE_MARGIN = 12             # after SWITCH_FRAME: window = running radius + LATE_MARGIN (px)
LATE_GROW_TOL = 1.06         # after SWITCH_FRAME the radius may not grow by more than 6 %
MAX_JUMP = 28                # largest centre move between frames (px); otherwise the frame is held
MIN_CIRC = 0.35              # smallest circularity of the hole region; otherwise the frame is held

# ---- acquisition (not used by the tracking, which works in pixels and frames) ----
PIXEL_SIZE_UM = 1.0
FRAME_INTERVAL_S = 7.0

# ---- checks ----
assert METHOD in ("halfway", "otsu"), METHOD
assert SEED in ("detect", "saved") and (SEED == "detect" or METHOD == "halfway"), SEED
DATA_DIR = DATA_ROOT / MEMBRANE_FOLDER
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Membrane data folder not found: {DATA_DIR}")
image_paths = sorted(DATA_DIR.glob(MEMBRANE_IMAGE_GLOB))
if not image_paths:
    raise FileNotFoundError(f"No {MEMBRANE_IMAGE_GLOB} files in {DATA_DIR}")
if SEED == "saved" and not (TRACKING_ROOT / MEMBRANE_FOLDER / "hole_tracking_results.csv").exists():
    raise FileNotFoundError(f'SEED = "saved" needs {TRACKING_ROOT / MEMBRANE_FOLDER / "hole_tracking_results.csv"}')
print(f"Found {len(image_paths)} membrane images in {DATA_DIR}; method '{METHOD}', seed '{SEED}' on frame {START_FRAME}")


## 3. Load the movie and check the seed

Loads every frame into one array (frames × height × width) and draws `START_FRAME` with the seed box `INIT_ROI`
next to the frame before it. **Check:** the wound is visible on `START_FRAME` and the yellow box contains the hole.
If not, change `START_FRAME` or `INIT_ROI` in section 1 and run again from there.

In [ ]:
# Load full time series into array shape (n_frames, height, width)
stack = np.stack([tifffile.imread(path) for path in image_paths], axis=0)
n_frames, height, width = stack.shape

print(f"Stack shape: {stack.shape}")
print(f"dtype: {stack.dtype}")
print(f"value range: [{stack.min():.4f}, {stack.max():.4f}]")
print(f"First file: {image_paths[0].name}")
print(f"Last file:  {image_paths[-1].name}")

In [ ]:
# The seed frame with the seed box: the box should contain the hole and as little else dark as possible.
from matplotlib.patches import Rectangle

if not 0 <= START_FRAME < n_frames:
    raise ValueError(f"START_FRAME = {START_FRAME}, but the movie has frames 0-{n_frames - 1}")
fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))
vmin, vmax = np.percentile(stack, [1, 99.5])
for ax, f in zip(axes, [max(START_FRAME - 1, 0), START_FRAME]):
    ax.imshow(stack[f], cmap="gray", vmin=vmin, vmax=vmax)
    if SEED == "detect":
        y0, y1, x0, x1 = INIT_ROI
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="yellow", lw=1.5))
    ax.set_title(f"frame {f}" + (" = START_FRAME" + (", INIT_ROI in yellow" if SEED == "detect" else "")
                                 if f == START_FRAME else " (one frame before START_FRAME)"), fontsize=10)
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
fig.tight_layout()
plt.show()


## 4. The two tracking methods

Both start from the seed: the largest dark region inside `INIT_ROI` on `START_FRAME` (or, with `SEED = "saved"`,
the circle of the supplied tracking on that frame). They differ in how they follow the hole in the frames after.

### Method `"otsu"` (and the seed detector of both methods)

Hole identification starts at `START_FRAME`, then tracks forward through the rest of the stack.

The hole is a compact, near-black disk that gradually shrinks and closes. We segment it with an Otsu threshold inside a **bounded search window** that follows the previous center:

1. **Seed** (`START_FRAME`): largest dark region inside `INIT_ROI` (`detect_dark_hole`, also the seed of the `"halfway"` method)
2. **Following frames**: Otsu segmentation inside a search window around the previous center; take the largest dark component and its darkness-weighted centroid
3. **Two-phase window**: a larger fixed window (`SEARCH_RADIUS_EARLY = 55`) while the hole is large (up to `SWITCH_FRAME`); after that the window **shrinks with the hole** (`running_radius + LATE_MARGIN`) and the radius is **not allowed to grow back** (capped at `running_radius × LATE_GROW_TOL`), since the hole only closes — this stops the outline from creeping onto the fuzzy radial streaks in the last frames
4. **Validity flag**: a frame is marked `held` (yellow overlay) when the detection jumps too far (`MAX_JUMP`) or the region gets too irregular (`MIN_CIRC`)

The **bounded window is deliberate**: the tissue around the hole is mottled and only slightly brighter, so an unbounded threshold bleeds outward and the measured area balloons frame after frame. The window keeps the region on the hole; the early size (55) avoids clipping the large early hole, and the shrinking late window follows the hole down to closure without re-growing.

Key parameters: `INIT_ROI`, `SWITCH_FRAME`, `SEARCH_RADIUS_EARLY`, `LATE_MARGIN`, `LATE_GROW_TOL`, `MAX_JUMP`, `MIN_CIRC`.

In [ ]:
from skimage.filters import gaussian, threshold_otsu
from skimage.measure import label, regionprops
from skimage.morphology import binary_closing, binary_opening, disk, remove_small_objects


def detect_dark_hole(img, roi_box, min_size=300):
    """Largest dark (Otsu) region inside roi_box, with a darkness-weighted centroid."""
    blur = gaussian(img, sigma=2.5, preserve_range=True)
    y0, y1, x0, x1 = roi_box
    roi = np.zeros(img.shape, dtype=bool)
    roi[y0:y1, x0:x1] = True

    thr = float(threshold_otsu(blur[y0:y1, x0:x1]))
    mask = (blur < thr) & roi
    mask = binary_opening(mask, disk(2))
    mask = binary_closing(mask, disk(8))
    mask = remove_small_objects(mask, min_size=min_size)

    props = regionprops(label(mask), blur)
    if not props:
        return None

    region = max(props, key=lambda r: r.area)

    region_mask = np.zeros(img.shape, dtype=bool)
    minr, minc, maxr, maxc = region.bbox
    region_mask[minr:maxr, minc:maxc] = region.image

    ys, xs = np.where(region.image)
    ys = ys + region.bbox[0]
    xs = xs + region.bbox[1]
    weights = np.clip(thr - blur[ys, xs], 0, None)
    cy = float(np.average(ys, weights=weights))
    cx = float(np.average(xs, weights=weights))
    circ = float(4 * np.pi * region.area / (region.perimeter ** 2 + 1e-6))

    return {
        "cy": cy,
        "cx": cx,
        "radius": float(region.equivalent_diameter / 2),
        "area": float(region.area),
        "circularity": circ,
        "mask": region_mask,
    }


def track_hole_stack(stack, start_frame=START_FRAME):
    """Seed the hole on start_frame, then track it forward inside a bounded window.

    Each state carries a 'valid' flag. It is set False when the detection jumps
    too far (`MAX_JUMP`) or the region becomes too irregular (`MIN_CIRC`) — i.e.
    the hole has closed / filled with radial streaks. In those cases the previous
    position and outline are held so the track is not corrupted.
    """
    init = detect_dark_hole(stack[start_frame], INIT_ROI)
    if init is None:
        raise RuntimeError(f"Could not detect hole on frame {start_frame}. Adjust INIT_ROI.")

    states = [None] * len(stack)
    init["valid"] = True
    states[start_frame] = init
    prev = init
    running_radius = init["radius"]  # tracks the (shrinking) hole size for the late phase

    for fi in range(start_frame + 1, len(stack)):
        cy0, cx0 = prev["cy"], prev["cx"]
        if fi < SWITCH_FRAME:
            search_radius = SEARCH_RADIUS_EARLY
        else:
            search_radius = int(running_radius + LATE_MARGIN)  # window shrinks with the hole
        y0 = max(0, int(cy0 - search_radius))
        y1 = min(stack.shape[1], int(cy0 + search_radius))
        x0 = max(0, int(cx0 - search_radius))
        x1 = min(stack.shape[2], int(cx0 + search_radius))

        det = detect_dark_hole(stack[fi], (y0, y1, x0, x1))

        held = {
            "cy": cy0, "cx": cx0, "radius": prev["radius"],
            "area": np.nan, "circularity": np.nan, "mask": prev.get("mask"),
        }
        valid = True
        if det is None:
            det = held
            valid = False
        else:
            jump = np.hypot(det["cy"] - cy0, det["cx"] - cx0)
            if jump > MAX_JUMP or det["circularity"] < MIN_CIRC:
                det = held
                valid = False
            elif fi >= SWITCH_FRAME and det["radius"] > running_radius * LATE_GROW_TOL:
                # the hole cannot re-open: cap the reported radius to the running size
                det["radius"] = running_radius

        # update the running radius (it only decreases in the late phase)
        if valid:
            if fi < SWITCH_FRAME:
                running_radius = det["radius"]
            else:
                running_radius = min(running_radius, det["radius"])

        det["valid"] = valid
        states[fi] = det
        prev = det

    return start_frame, states

### Method `"halfway"`

Starting from the seed circle, in every frame from `START_FRAME` on (`detect_hole_halfway`), with the previous
frame's circle:
1. Blur the frame (`TRACK_SIGMA`).
2. **Hole floor:** 10th percentile inside 0.7 × the previous radius. **Tissue:** median in a ring 8-30 px outside
   the previous edge (`TRACK_TISSUE_RING`).
3. **Hole:** pixels darker than halfway between floor and tissue (`TRACK_LEVEL`), inside a window of the previous
   radius + max(0.3 × radius, 6 px) (`TRACK_WINDOW`), cleaned with a small opening and closing. The piece that
   contains the previous centre (or else the nearest piece) is the hole.
4. **Circle:** centre = the piece's centroid; radius = √(area / π).
5. If no piece is found, or the centre moves more than `TRACK_MAX_JUMP`, the previous circle is held (the frame
   is marked as held).

Because the floor and the tissue level are measured again in every frame, the edge stays on the hole's own
contrast even when the filling hole is only a little darker than the tissue. Each frame is first capped at
`CLIP_PERCENTILE`, so that a very bright spot next to the wound does not set the seed threshold.

In [ ]:
# ---- frame-by-frame tracking: hole edge halfway between the hole floor and the tissue ----
from skimage.morphology import opening, closing


def detect_hole_halfway(img, prev):
    # prev = (cx, cy, r) of the previous frame; returns the hole in this frame, or None
    cx0, cy0, r0 = prev
    blur = gaussian(img, sigma=TRACK_SIGMA, preserve_range=True)
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    d = np.hypot(xx - cx0, yy - cy0)
    floor = np.percentile(blur[d < max(0.7 * r0, 4)], 10)
    tissue = np.median(blur[(d > r0 + TRACK_TISSUE_RING[0]) & (d < r0 + TRACK_TISSUE_RING[1])])
    thr = floor + TRACK_LEVEL * (tissue - floor)
    window = d < r0 + max(TRACK_WINDOW[0] * r0, TRACK_WINDOW[1])
    lab = label(closing(opening((blur < thr) & window, disk(1)), disk(3)))
    best, best_d = None, np.inf
    for rp in regionprops(lab):
        if rp.area < 20:
            continue
        cy, cx = rp.centroid
        dist = -1.0 if lab[int(round(cy0)), int(round(cx0))] == rp.label else np.hypot(cx - cx0, cy - cy0)
        if dist < best_d:
            best, best_d = rp, dist
    if best is None or best_d > TRACK_MAX_JUMP:
        return None
    cy, cx = best.centroid
    return {"cy": float(cy), "cx": float(cx), "radius": float(np.sqrt(best.area / np.pi)),
            "contrast": float((tissue - floor) / tissue), "mask": lab == best.label}


def track_closing_hole(stack, start_frame, init_roi=None, seed=None):
    # seed = (cx, cy, r), or found with the old detector inside init_roi; then follow the hole frame by frame.
    # Returns states[frame] (None before start_frame).
    if seed is None:
        det = detect_dark_hole(stack[start_frame], init_roi)
        if det is None:
            raise RuntimeError(f"Could not detect the hole on frame {start_frame}. Adjust INIT_ROI.")
        seed = (det["cx"], det["cy"], det["radius"])
    prev = tuple(seed)
    states = [None] * len(stack)
    for fi in range(start_frame, len(stack)):
        det = detect_hole_halfway(stack[fi], prev)
        if det is None:
            det = {"cx": prev[0], "cy": prev[1], "radius": prev[2], "contrast": np.nan, "mask": None, "valid": False}
        else:
            det["valid"] = True
        states[fi] = det
        prev = (det["cx"], det["cy"], det["radius"])
    return states

## 5. Run the tracking

Runs the chosen method from `START_FRAME` to the last frame and prints the frames tracked, the frames in which the
hole was found (not held), and the seed circle.

In [ ]:
# Run detection + tracking from START_FRAME onward
hole_cy = np.full(n_frames, np.nan)
hole_cx = np.full(n_frames, np.nan)
hole_r = np.full(n_frames, np.nan)
hole_valid = np.zeros(n_frames, dtype=bool)
hole_masks = [None] * n_frames  # real segmented outline of the hole per frame

if METHOD == "otsu":
    init_frame, hole_states = track_hole_stack(stack)
else:   # "halfway": 64-bit floats, every frame capped at CLIP_PERCENTILE
    stack64 = stack.astype(float)
    clipped = np.minimum(stack64, np.percentile(stack64, CLIP_PERCENTILE, axis=(1, 2), keepdims=True))
    init_frame = START_FRAME
    if SEED == "saved":   # circle of the supplied tracking on START_FRAME
        sup = pd.read_csv(TRACKING_ROOT / MEMBRANE_FOLDER / "hole_tracking_results.csv").set_index("frame")
        seed = tuple(float(sup.loc[START_FRAME, k]) for k in ("cx", "cy", "radius"))
        hole_states = track_closing_hole(clipped, init_frame, seed=seed)
    else:
        hole_states = track_closing_hole(clipped, init_frame, init_roi=INIT_ROI)

for fi, state in enumerate(hole_states):
    if state is not None:
        hole_cy[fi] = state["cy"]
        hole_cx[fi] = state["cx"]
        hole_r[fi] = state["radius"]
        hole_valid[fi] = state["valid"]
        hole_masks[fi] = state.get("mask")

tracked_frames = np.where(~np.isnan(hole_cy))[0]
valid_frames = np.where(hole_valid)[0]
print(f"Hole identification starts at frame {init_frame}")
print(f"Tracked frames: {tracked_frames[0]}\u2013{tracked_frames[-1]} ({len(tracked_frames)} frames)")
print(f"Valid (hole open) frames: {valid_frames[0]}\u2013{valid_frames[-1]} ({len(valid_frames)} frames)")
print(
    f"Frame {init_frame}: center=({hole_cy[init_frame]:.1f}, {hole_cx[init_frame]:.1f}), "
    f"radius={hole_r[init_frame]:.1f} px"
)

## 6. Check the result

- **Outline overlay** (all frames, zoomed on the hole): the real outline of the detected hole (cyan) and the
  fitted circle (dashed: green where the hole was found, yellow where the previous circle was held; the panel
  title says *open* or *closed*). The circle should sit on the hole edge in every frame.
- **Hole centre and radius against frame:** the radius should shrink smoothly as the wound closes, and the centre
  should not jump.

If the outline leaves the hole, see section 8.

In [ ]:
from matplotlib.patches import Circle
from skimage.measure import find_contours

# Overlay real outline (cyan) + fitted circle (green open / yellow closed) — all frames.
# Each panel is zoomed to the hole region so the outline/circle are clearly visible.
overlay_frames = np.arange(START_FRAME, n_frames)
ncols = 5
nrows = int(np.ceil(len(overlay_frames) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4.2 * nrows))
axes = np.atleast_1d(axes).ravel()

vmin, vmax = np.percentile(stack, [1, 99])

# zoom window around the hole (padding beyond the largest detected radius)
valid_r = hole_r[~np.isnan(hole_r)]
pad = float(np.nanmax(valid_r)) + 35 if valid_r.size else 90
cy_mid = float(np.nanmedian(hole_cy))
cx_mid = float(np.nanmedian(hole_cx))
y_lo, y_hi = cy_mid - pad, cy_mid + pad
x_lo, x_hi = cx_mid - pad, cx_mid + pad

for ax, fi in zip(axes, overlay_frames):
    ax.imshow(stack[fi], cmap="gray", origin="upper", vmin=vmin, vmax=vmax)

    # real segmented outline of the hole
    mask = hole_masks[fi]
    if mask is not None and mask.any():
        for contour in find_contours(mask.astype(float), 0.5):
            ax.plot(contour[:, 1], contour[:, 0], color="cyan", linewidth=2)

    # fitted circle
    color = "lime" if hole_valid[fi] else "yellow"
    ax.add_patch(
        Circle((hole_cx[fi], hole_cy[fi]), hole_r[fi], fill=False,
               edgecolor=color, linewidth=2.2, linestyle="--")
    )
    ax.plot(hole_cx[fi], hole_cy[fi], "r+", markersize=11, mew=2)
    status = "open" if hole_valid[fi] else "closed"
    ax.set_title(f"f{fi} ({status})", fontsize=12)
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(y_hi, y_lo)  # keep image (top-down) orientation
    ax.axis("off")

for ax in axes[len(overlay_frames):]:
    ax.axis("off")

fig.suptitle("Hole: real outline (cyan) vs fitted circle (dashed) — all frames", y=1.005, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Hole geometry vs time (from START_FRAME on)
frames = np.arange(START_FRAME, n_frames)
cy_plot = hole_cy[START_FRAME:]
cx_plot = hole_cx[START_FRAME:]
r_plot = hole_r[START_FRAME:]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(frames, cy_plot, "-o", markersize=3)
axes[0].set_xlabel("Frame index")
axes[0].set_ylabel("cy (px)")
axes[0].set_title("Hole center Y")
axes[0].grid(True, alpha=0.3)

axes[1].plot(frames, cx_plot, "-o", markersize=3, color="tab:orange")
axes[1].set_xlabel("Frame index")
axes[1].set_ylabel("cx (px)")
axes[1].set_title("Hole center X")
axes[1].grid(True, alpha=0.3)

axes[2].plot(frames, r_plot, "-o", markersize=3, color="tab:green")
axes[2].set_xlabel("Frame index")
axes[2].set_ylabel("radius (px)")
axes[2].set_title("Hole radius")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Trajectory in image coordinates
plt.figure(figsize=(6, 6))
plt.plot(cx_plot, cy_plot, "-o", markersize=3)
plt.gca().invert_yaxis()
plt.xlabel("cx (px)")
plt.ylabel("cy (px)")
plt.title("Hole trajectory (image coordinates)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Save

Writes `frame, cy, cx, radius` for every frame (NaN before `START_FRAME`) to
`wound_tracking/<MEMBRANE_FOLDER>/<HOLE_TRACKING_CSV>`.

In [ ]:
# Save results table (NaN for frames before START_FRAME)
all_frames = np.arange(n_frames)
(TRACKING_ROOT / MEMBRANE_FOLDER).mkdir(parents=True, exist_ok=True)
out_csv = TRACKING_ROOT / MEMBRANE_FOLDER / HOLE_TRACKING_CSV
np.savetxt(
    out_csv,
    np.column_stack([all_frames, hole_cy, hole_cx, hole_r]),
    delimiter=",",
    header="frame,cy,cx,radius",
    comments="",
)
print(f"Saved: {out_csv}")

## 8. Troubleshooting

| Problem | What to change |
|---|---|
| `Could not detect the hole on frame ...` | The seed box holds no dark region on `START_FRAME`: check the figure in section 3; move or enlarge `INIT_ROI`, or choose a later `START_FRAME`. |
| The seed sits on another dark object | Make `INIT_ROI` smaller and tighter around the hole. |
| `"halfway"`: the outline jumps to a neighbouring dark area, or many frames are held | Lower `TRACK_MAX_JUMP` to stop jumps, raise it if the wound really moves; raise `TRACK_SIGMA` for noisy images. |
| `"halfway"`: a bright spot next to the wound pulls the seed | Lower `CLIP_PERCENTILE` (e.g. 90). |
| `"otsu"`: the outline spreads beyond the hole into the tissue | Lower `SEARCH_RADIUS_EARLY` or `LATE_MARGIN`, or set `SWITCH_FRAME` earlier; or use `"halfway"`. |
| `"otsu"`: the radius stops shrinking while the hole still closes | The hole has become too faint for one Otsu threshold: use `"halfway"`. |
| The wound is not visible as a dark hole in any channel | This notebook cannot track it. Write your own tracking file with the columns `frame, cy, cx, radius` (pixels, one row per frame, NaN where unknown) and use it in notebook 2. |

## Appendix: settings used for the paper's datasets

**`"halfway"` datasets** (all with the method settings of section 2). Four of them start from their supplied
circle (`SEED = "saved"`), because their supplied tracking follows the hole correctly at first and only stops near
the end. ctrl_0820's supplied tracking never found the hole, so its seed is detected.

| Dataset | `MEMBRANE_FOLDER` | `SEED` | `START_FRAME` | `INIT_ROI` |
|---|---|---|---|---|
| ctrl_0808 | ctrl_0808_membrane | `"saved"` | 3 | - |
| ctrl_0820 | ctrl_0820_membrane | `"detect"` | 21 | (140, 270, 225, 360) |
| Gd3-4 | Gd3-4-membrane | `"saved"` | 3 | - |
| CBX_6 | CBX_6_membrane | `"saved"` | 22 | - |
| hep_6 | hep_6_membrane | `"saved"` | 22 | - |

**`"otsu"` datasets** (pom1-3, Gd3_3, CBX_5, CBX_7, hep_7, heptanol_3, heptanol_5): their tracking is supplied in
`wound_tracking/<membrane folder>/hole_tracking_results.csv`. Their per-dataset settings (`START_FRAME`,
`INIT_ROI`, `SWITCH_FRAME`, window sizes) were tuned by hand and not recorded.